<a href="https://colab.research.google.com/github/danimeb/big-data-team/blob/main/proyecto-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Proyecto Etapa 2. Selección de técnica de muestreo para la construcción de muestra inicial**

**Equipo 04**

- Daniela Montserrat Enríquez Ballesteros (A01797865)

- Humberto Alejandro Espinola Gonzalez (A01840066)

- Juan Paulo Pérez Tejada Ladrón de Guevara (A01797972)

- Mónica Raquel Saucedo Alcalá (A01797710)

## Descripción de la base de datos

### Contexto

La Comisión de Taxis y Limusinas (TLC) de la Ciudad de Nueva York (NYC) almacena datos de todos sus taxis, disponibles gratuitamente en su sitio oficial. Puedes acceder a ellos [aquí](https://www1.nyc.gov/site/tlc/about/tlc-trip-record-data.page).

- En este dataset se consideran únicamente los datos de **Taxis Amarillos**, correspondientes a **enero de 2015 y enero-marzo de 2016**, disponibles en kaggle en [este enlace](https://www.kaggle.com/datasets/elemento/nyc-yellow-taxi-trip-data). Contiene 47,248,845 filas con 19 atributos (7.39 GB en total), distribuidos en 4 archivos csv.


## Atributos

| Nombre del Campo | Descripción |
|---|---|
| VendorID | Código que identifica al proveedor TPEP que generó el registro.<br><br>1. Creative Mobile Technologies<br>2. VeriFone Inc. |
| tpep_pickup_datetime | Fecha y hora en que se activó el taxímetro. |
| tpep_dropoff_datetime | Fecha y hora en que se desactivó el taxímetro. |
| Passenger_count | Número de pasajeros en el vehículo. Valor ingresado por el conductor. |
| Trip_distance | Distancia recorrida en millas, reportada por el taxímetro. |
| Pickup_longitude | Longitud del punto de recogida. |
| Pickup_latitude | Latitud del punto de recogida. |
| RateCodeID | Código de tarifa vigente al finalizar el viaje.<br><br>1. Tarifa estándar<br>2. JFK<br>3. Newark<br>4. Nassau o Westchester<br>5. Tarifa negociada<br>6. Viaje compartido |
| Store_and_fwd_flag | Indica si el registro del viaje se almacenó en la memoria del vehículo antes de enviarse al proveedor (es decir, "store and forward"), debido a falta de conexión al servidor.<br>Y = viaje almacenado y reenviado<br>N = viaje no almacenado |
| Dropoff_longitude | Longitud del punto de entrega. |
| Dropoff_latitude | Latitud del punto de entrega. |
| Payment_type | Código numérico que indica el método de pago del pasajero.<br><br>1. Tarjeta de crédito<br>2. Efectivo<br>3. Sin cargo<br>4. Disputa<br>5. Desconocido<br>6. Viaje anulado |
| Fare_amount | Tarifa calculada por el taxímetro en función del tiempo y la distancia. |
| Extra | Cargos adicionales y recargos varios. Actualmente incluye los cargos de $0.50 y $1 por hora pico y servicio nocturno. |
| MTA_tax | Impuesto MTA de $0.50, aplicado automáticamente según la tarifa del taxímetro en uso. |
| Improvement_surcharge | Recargo de mejora de $0.30, aplicado al inicio del viaje. Este recargo comenzó a cobrarse en 2015. |
| Tip_amount | Monto de la propina. Este campo se completa automáticamente para propinas pagadas con tarjeta de crédito. Las propinas en efectivo no se incluyen. |
| Tolls_amount | Monto total de peajes pagados durante el viaje. |
| Total_amount | Monto total cobrado al pasajero. No incluye propinas en efectivo. |

### Exploración inicial

En la exploración inicial del dataset, se observó gran variación de acuerdo a tres categorías: hora, zona de recogida y día de la semana. Los viernes y sábados se observa un pico en la noche, mientras que los domingos son los días con menos corridas.

## Inicialización y Carga de Datos

Esta sección instala las dependencias requeridas y descarga los datos de Kaggle.

In [2]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget -q https://dlcdn.apache.org/spark/spark-3.5.8/spark-3.5.8-bin-hadoop3.tgz
!tar -xzf spark-3.5.8-bin-hadoop3.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.8-bin-hadoop3"

import findspark
findspark.init()

from pyspark.sql import SparkSession

# Create SparkSession after all environment setup
spark = SparkSession.builder \
    .appName("nyc_taxi_eda") \
    .config("spark.sql.session.timeZone", "UTC") \
    .getOrCreate()

In [3]:
import kagglehub
# Set the path to the file you'd like to load
file_path = kagglehub.dataset_download("elemento/nyc-yellow-taxi-trip-data")

print("Path to dataset files:", file_path)

Using Colab cache for faster access to the 'nyc-yellow-taxi-trip-data' dataset.
Path to dataset files: /kaggle/input/nyc-yellow-taxi-trip-data


In [4]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, TimestampType, FloatType, BooleanType

# Define the schema
schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", IntegerType(), True),
    StructField("trip_distance", FloatType(), True),
    StructField("pickup_longitude", FloatType(), True),
    StructField("pickup_latitude", FloatType(), True),
    StructField("RateCodeID", IntegerType(), True),
    StructField("store_and_fwd_flag", BooleanType(), True),
    StructField("dropoff_longitude", FloatType(), True),
    StructField("dropoff_latitude", FloatType(), True),
    StructField("payment_type", IntegerType(), True),
    StructField("fare_amount", FloatType(), True),
    StructField("extra", FloatType(), True),
    StructField("mta_tax", FloatType(), True),
    StructField("tip_amount", FloatType(), True),
    StructField("tolls_amount", FloatType(), True),
    StructField("improvement_surcharge", FloatType(), True),
    StructField("total_amount", FloatType(), True)
])

# Spark reads the 4 CSVs with the predefined schema
df = spark.read.csv(
    file_path + "/*.csv",
    header=True,
    schema=schema, # Use the defined schema
    timestampFormat="yyyy-MM-dd HH:mm:ss"
)

print(f"Filas totales: {df.count():,}")
df.printSchema()

Filas totales: 47,248,845
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: float (nullable = true)
 |-- pickup_longitude: float (nullable = true)
 |-- pickup_latitude: float (nullable = true)
 |-- RateCodeID: integer (nullable = true)
 |-- store_and_fwd_flag: boolean (nullable = true)
 |-- dropoff_longitude: float (nullable = true)
 |-- dropoff_latitude: float (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: float (nullable = true)
 |-- extra: float (nullable = true)
 |-- mta_tax: float (nullable = true)
 |-- tip_amount: float (nullable = true)
 |-- tolls_amount: float (nullable = true)
 |-- improvement_surcharge: float (nullable = true)
 |-- total_amount: float (nullable = true)



## Limpieza de la base de datos (pre-procesamiento)

Antes de particionar se aplican los filtros de calidad identificados en la **Etapa 1**. Esto es necesario porque los registros inválidos (coordenadas en (0,0), montos negativos, distancias imposibles, etc.) distorsionarían las particiones; en particular, las ~1.5 M de coordenadas (0,0) caerían todas dentro de la zona `outer` y sesgarían esa partición, además de inflar los promedios de distancia.

| Problema | Regla de filtrado |
|---|---|
| Coordenadas (0,0) y fuera de NYC | bounding box: lat [40.5, 41.0], lon [-74.3, -73.7] |
| Distancia inválida | `trip_distance` en el rango (0, 100] millas |
| Duración inválida | `dropoff > pickup` y duración ≤ 6 h |
| Montos no válidos | `fare_amount > 0` y `total_amount > 0` |
| Pasajeros inválidos | `passenger_count` en [1, 6] |

A partir de aquí se trabaja sobre la base **limpia** `D`.

In [5]:
# --- Limpieza de D (filtros de calidad de la Etapa 1) ---

# Duración del viaje en minutos (variable derivada usada para validar)
df = df.withColumn(
    "trip_duration_min",
    (F.col("tpep_dropoff_datetime").cast("long") -
     F.col("tpep_pickup_datetime").cast("long")) / 60.0
)

df = df.filter(
    # Filtrado geográfico: bounding box de NYC (elimina (0,0) y fuera de área)
    F.col("pickup_latitude").between(40.5, 41.0) &
    F.col("pickup_longitude").between(-74.3, -73.7) &
    # Distancia válida
    (F.col("trip_distance") > 0) & (F.col("trip_distance") <= 100) &
    # Duración válida
    (F.col("trip_duration_min") > 0) & (F.col("trip_duration_min") <= 360) &
    # Integridad financiera
    (F.col("fare_amount") > 0) & (F.col("total_amount") > 0) &
    # Validación categórica
    F.col("passenger_count").between(1, 6)
)

# Cache: D limpia se reutiliza en todas las particiones
df = df.cache()

n_limpio = df.count()
print(f"Filas tras limpieza (D limpia) : {n_limpio:,}")


Filas tras limpieza (D limpia) : 46,152,731


NameError: name 'n_raw' is not defined

## Reglas de particionamiento

El dataset `D` se particiona usando **tres variables de caracterización**:

### Variable A — Franja horaria (`franja_horaria`)

| Valor | Rango (hora local) | P(A) |
|---|---|---|
| `madrugada` | 0 – 5 h | 0.08 |
| `mañana` | 6 – 11 h | 0.22 |
| `tarde` | 12 – 17 h | 0.28 |
| `noche` | 18 – 23 h | 0.42 |

### Variable B — Zona geográfica de recogida (`zona_geo`)

| Valor | Descripción | P(B) |
|---|---|---|
| `manhattan` | Lat [40.700, 40.878], Lon [-74.020, -73.907] | 0.65 |
| `jfk` | Lat [40.620, 40.665], Lon [-73.815, -73.748] | 0.10 |
| `laguardia` | Lat [40.760, 40.795], Lon [-73.895, -73.855] | 0.08 |
| `outer` | Resto del área operativa de NYC | 0.17 |

### Variable C — Tipo de día (`tipo_dia`)

| Valor | Días | P(C) |
|---|---|---|
| `entre_semana` | Lunes – Jueves | 0.53 |
| `fin_de_semana` | Viernes – Sábado | 0.32 |
| `valle` | Domingo | 0.15 |

### Combinaciones

Bajo el supuesto de independencia entre variables, se generan **4 × 4 × 3 = 48 particiones**.
La probabilidad de cada partición es:

$$P(A_i, B_j, C_k) = P(A_i) \times P(B_j) \times P(C_k)$$

La partición de mayor peso es `noche × manhattan × entre_semana` con $P \approx 0.145$ (~14.5 % de los registros).

## Creación de Variables de Caracterización

In [6]:
df = df.withColumn("pickup_weekday", F.dayofweek("tpep_pickup_datetime"))
df = df.withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))


# Variable A: franja_horaria
df = df.withColumn(
    "franja_horaria",
    F.when(F.col("pickup_hour").between(0, 5),   "madrugada")
     .when(F.col("pickup_hour").between(6, 11),  "manana")
     .when(F.col("pickup_hour").between(12, 17), "tarde")
     .otherwise("noche")
)

# Variable B: zona_geo
df = df.withColumn(
    "zona_geo",
    F.when(
        F.col("pickup_latitude").between(40.700, 40.878) &
        F.col("pickup_longitude").between(-74.020, -73.907),
        "manhattan"
    ).when(
        F.col("pickup_latitude").between(40.620, 40.665) &
        F.col("pickup_longitude").between(-73.815, -73.748),
        "jfk"
    ).when(
        F.col("pickup_latitude").between(40.760, 40.795) &
        F.col("pickup_longitude").between(-73.895, -73.855),
        "laguardia"
    ).otherwise("outer")
)

# Variable C: tipo_dia  (dayofweek: 1=Dom, 2=Lun, 3=Mar, 4=Mie, 5=Jue, 6=Vie, 7=Sab)
df = df.withColumn(
    "tipo_dia",
    F.when(F.col("pickup_weekday").isin([6, 7]), "fin_de_semana")   # Vie, Sab
     .when(F.col("pickup_weekday") == 1,          "valle")           # Dom
     .otherwise("entre_semana")                                      # Lun-Jue
)

# Columna combinada de estrato
df = df.withColumn(
    "estrato",
    F.concat_ws("__", "franja_horaria", "zona_geo", "tipo_dia")
)

print("Variables de particionamiento asignadas.")
df.select("tpep_pickup_datetime", "franja_horaria", "zona_geo", "tipo_dia", "estrato").show(10, truncate=False)


Variables de particionamiento asignadas.
+--------------------+--------------+---------+-------------+-------------------------------+
|tpep_pickup_datetime|franja_horaria|zona_geo |tipo_dia     |estrato                        |
+--------------------+--------------+---------+-------------+-------------------------------+
|2015-01-15 19:05:39 |noche         |manhattan|entre_semana |noche__manhattan__entre_semana |
|2015-01-10 20:33:38 |noche         |manhattan|fin_de_semana|noche__manhattan__fin_de_semana|
|2015-01-10 20:33:38 |noche         |manhattan|fin_de_semana|noche__manhattan__fin_de_semana|
|2015-01-10 20:33:39 |noche         |manhattan|fin_de_semana|noche__manhattan__fin_de_semana|
|2015-01-10 20:33:39 |noche         |manhattan|fin_de_semana|noche__manhattan__fin_de_semana|
|2015-01-10 20:33:39 |noche         |laguardia|fin_de_semana|noche__laguardia__fin_de_semana|
|2015-01-10 20:33:39 |noche         |manhattan|fin_de_semana|noche__manhattan__fin_de_semana|
|2015-01-10 20:33:3

## Extracción de Submuestras por Partición



In [7]:
def resumen_particion(df_muestra, nombre):
    """Imprime estadísticas básicas de una submuestra."""
    n = df_muestra.count()
    print(f"{'='*60}")
    print(f"Partición : {nombre}")
    print(f"Registros : {n:,}")
    if n > 0:
        df_muestra.select(
            F.round(F.mean("trip_distance"), 2).alias("dist_media_mi"),
            F.round(F.mean("fare_amount"), 2).alias("tarifa_media"),
            F.round(F.mean("passenger_count"), 2).alias("pasajeros_media")
        ).show(truncate=False)
    print()


### Partición: `noche × manhattan × entre_semana`
> Partición de mayor peso teórico (~14.5 %)

In [8]:
p1 = df.filter(
    (F.col("franja_horaria") == "noche") &
    (F.col("zona_geo")       == "manhattan") &
    (F.col("tipo_dia")       == "entre_semana")
)


In [9]:
p1.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------------+---------------+----------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+------------------+--------------+-----------+--------------+---------+------------+------------------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|pickup_longitude|pickup_latitude|RateCodeID|store_and_fwd_flag|dropoff_longitude|dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|trip_duration_min |pickup_weekday|pickup_hour|franja_horaria|zona_geo |tipo_dia    |estrato                       |
+--------+--------------------+---------------------+---------------+-------------+----------------+---------------+----------+------------------+-----------------+----------------+------------+-----------+----

In [10]:
resumen_particion(p1, "noche × manhattan × entre_semana")


Partición : noche × manhattan × entre_semana
Registros : 8,424,733
+-------------+------------+---------------+
|dist_media_mi|tarifa_media|pasajeros_media|
+-------------+------------+---------------+
|2.44         |10.86       |1.65           |
+-------------+------------+---------------+




### Partición: `tarde × manhattan × entre_semana`
> Segunda partición en peso teórico (~9.6 %)

In [11]:
p2 = df.filter(
    (F.col("franja_horaria") == "tarde") &
    (F.col("zona_geo")       == "manhattan") &
    (F.col("tipo_dia")       == "entre_semana")
)
p2.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------------+---------------+----------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+------------------+--------------+-----------+--------------+---------+------------+------------------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|pickup_longitude|pickup_latitude|RateCodeID|store_and_fwd_flag|dropoff_longitude|dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|trip_duration_min |pickup_weekday|pickup_hour|franja_horaria|zona_geo |tipo_dia    |estrato                       |
+--------+--------------------+---------------------+---------------+-------------+----------------+---------------+----------+------------------+-----------------+----------------+------------+-----------+----

In [12]:
resumen_particion(p2, "tarde × manhattan × entre_semana")


Partición : tarde × manhattan × entre_semana
Registros : 7,086,438
+-------------+------------+---------------+
|dist_media_mi|tarifa_media|pasajeros_media|
+-------------+------------+---------------+
|2.2          |11.17       |1.65           |
+-------------+------------+---------------+




### Partición: `noche × manhattan × fin_de_semana`
> Viernes y sábado nocturnos en Manhattan (~8.7 %)

In [13]:
p3 = df.filter(
    (F.col("franja_horaria") == "noche") &
    (F.col("zona_geo")       == "manhattan") &
    (F.col("tipo_dia")       == "fin_de_semana")
)
p3.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------------+---------------+----------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+------------------+--------------+-----------+--------------+---------+-------------+-------------------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|pickup_longitude|pickup_latitude|RateCodeID|store_and_fwd_flag|dropoff_longitude|dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|trip_duration_min |pickup_weekday|pickup_hour|franja_horaria|zona_geo |tipo_dia     |estrato                        |
+--------+--------------------+---------------------+---------------+-------------+----------------+---------------+----------+------------------+-----------------+----------------+------------+-----------+

In [14]:
resumen_particion(p3, "noche × manhattan × fin_de_semana")


Partición : noche × manhattan × fin_de_semana
Registros : 4,934,299
+-------------+------------+---------------+
|dist_media_mi|tarifa_media|pasajeros_media|
+-------------+------------+---------------+
|2.3          |10.83       |1.73           |
+-------------+------------+---------------+




## Confirmación de cobertura

Verificar que los 48 estratos están presentes

In [15]:
cobertura = (df.groupBy("estrato").count()
             .orderBy(F.col("count").desc()))

total_estratos = cobertura.count()
total_registros = cobertura.agg(F.sum("count")).collect()[0][0]

print(f"Estratos distintos encontrados : {total_estratos} (esperado: 48)")
print(f"Total de registros cubiertos   : {total_registros:,}")
print()
cobertura.show(48, truncate=False)

Estratos distintos encontrados : 48 (esperado: 48)
Total de registros cubiertos   : 46,152,731

+-----------------------------------+-------+
|estrato                            |count  |
+-----------------------------------+-------+
|noche__manhattan__entre_semana     |8424733|
|tarde__manhattan__entre_semana     |7086438|
|manana__manhattan__entre_semana    |6694636|
|noche__manhattan__fin_de_semana    |4934299|
|tarde__manhattan__fin_de_semana    |4075268|
|manana__manhattan__fin_de_semana   |2913607|
|madrugada__manhattan__fin_de_semana|2079127|
|tarde__manhattan__valle            |1860928|
|madrugada__manhattan__entre_semana |1665637|
|noche__manhattan__valle            |1442626|
|madrugada__manhattan__valle        |1321121|
|manana__manhattan__valle           |964846 |
|noche__laguardia__entre_semana     |257462 |
|tarde__laguardia__entre_semana     |224847 |
|noche__jfk__entre_semana           |194753 |
|tarde__jfk__entre_semana           |175651 |
|manana__laguardia__entre_sema

In [ ]:
spark.stop()